In [ ]:
# Data loading script for engine sensor data

import pandas as pd

COLUMNS = [
    "engine_id", "cycle",
    "op_setting_1", "op_setting_2", "op_setting_3"
] + [f"sensor_{i}" for i in range(1, 22)]

def load_data(path):
    df = pd.read_csv(path, sep=" ", header=None)
    df = df.iloc[:, :26]
    df.columns = COLUMNS
    return df

if __name__ == "__main__":
    df = load_data("data/train_FD001.csv")
    print(df.head())

In [ ]:
# Feature Engineering

import pandas as pd

def add_remaining_useful_life(df):
    max_cycle = df.groupby("engine_id")["cycle"].max()
    df["RUL"] = df.apply(
        lambda row: max_cycle[row["engine_id"]] - row["cycle"], axis=1
    )
    return df

def rolling_features(df, window=5):
    for col in ["sensor_2", "sensor_7", "sensor_12"]:
        df[f"{col}_rolling_mean"] = (
            df.groupby("engine_id")[col]
            .rolling(window)
            .mean()
            .reset_index(0, drop=True)
        )
    return df

if __name__ == "__main__":
    df = pd.read_csv("data/train_FD001_processed.csv")
    df = add_remaining_useful_life(df)
    df = rolling_features(df)
    df.to_csv("data/train_FD001_features.csv", index=False)

In [ ]:
# Model Training Script

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

df = pd.read_csv("data/train_FD001_features.csv").dropna()

FEATURES = [col for col in df.columns if "sensor" in col]
X = df[FEATURES]
y = df["RUL"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

preds = model.predict(X_test)
mae = mean_absolute_error(y_test, preds)

print("MAE:", mae)

In [ ]:
# Data Visualization Script

import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("data/train_FD001_features.csv")

engine_sample = df[df["engine_id"] == 1]

plt.figure(figsize=(10,4))
plt.plot(engine_sample["cycle"], engine_sample["sensor_2"])
plt.title("Sensor 2 Degradation Over Time (Engine 1)")
plt.xlabel("Cycle")
plt.ylabel("Sensor Value")
plt.tight_layout()
plt.savefig("screenshots/sensor_degradation.png")
plt.close()

plt.figure(figsize=(6,4))
plt.hist(df["RUL"], bins=30)
plt.title("Remaining Useful Life Distribution")
plt.xlabel("RUL")
plt.tight_layout()
plt.savefig("screenshots/rul_distribution.png")
plt.close()
